# Section 1: Project Information

**KidsNutriBite Evaluation Playground**
This notebook serves as the official experimental environment for the KidsNutriBite project. It orchestrates a Hybrid RAG pipeline and evaluates responses using a fully decoupled, deterministic Layer 2 metrics architecture.

**Workflow:**
Query -> Retriever -> Planner -> LLM Generator -> Layer 1 (Semantic Judges) -> Layer 2 (Deterministic Metrics) -> Reports.

## Section 2: Environment Setup
Install all required dependencies. Restart the runtime after execution if prompted.

In [ ]:
!pip install torch transformers sentence-transformers faiss-cpu bitsandbytes accelerate python-dotenv groq google-generativeai matplotlib pandas numpy scikit-learn huggingface_hub tabulate

## Section 3: HuggingFace Login
Authenticate with HuggingFace Hub to download required models (e.g., Llama 3 or Qwen local weights).

In [ ]:
import os
from huggingface_hub import login

if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
    from kaggle_secrets import UserSecretsClient
    try:
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
        login(token=hf_token)
        print("Successfully logged into HuggingFace Hub via Kaggle Secrets.")
    except Exception as e:
        print("HF_TOKEN not found in Kaggle Secrets.")
else:
    print("To login to HuggingFace, uncomment the line below and provide your token.")
    # login(token="YOUR_HF_TOKEN")


## Section 4: API Keys
Configure Gemini (Judge & Generator) and OpenRouter (Optional Generator) API keys.

In [ ]:
import os
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    try:
        os.environ['GEMINI_API_KEY'] = secrets.get_secret('GEMINI_API_KEY')
        os.environ['OPENROUTER_API_KEY'] = secrets.get_secret('OPENROUTER_API_KEY')
        print("Loaded APIs from Kaggle Secrets.")
    except:
        pass
else:
    os.environ["GEMINI_API_KEY"] = "YOUR_GEMINI_API_KEY"
    os.environ["OPENROUTER_API_KEY"] = "YOUR_OPENROUTER_API_KEY"

with open(".env", "w") as f:
    f.write(f"GEMINI_API_KEY=\"{os.environ.get('GEMINI_API_KEY', '')}\"\n")
    f.write(f"OPENROUTER_API_KEY=\"{os.environ.get('OPENROUTER_API_KEY', '')}\"\n")


## Section 5: Dataset Loading
Verify all structural dataset JSONs are present and log their row counts.

In [ ]:
import json, os

data_dir = os.path.join("data", "planner")
files = {"Foods": "foods.json", "Conditions": "conditions.json", "Goals": "goals.json", "Allergies": "allergies.json"}

for name, file in files.items():
    path = os.path.join(data_dir, file)
    if os.path.exists(path):
        with open(path, "r") as f:
            data = json.load(f)
            print(f"{name} loaded successfully: {len(data)} records.")
    else:
        print(f"Missing: {path}")

rag_path = os.path.join("data", "rag", "rag_data.json")
if os.path.exists(rag_path):
    with open(rag_path, "r") as f:
        rag_data = json.load(f)
        print(f"RAG Data loaded successfully: {len(rag_data)} chunks.")


## Section 6: Load FAISS
Build the FAISS index if missing, and initialize the `KidsNutriRetriever`.

In [ ]:
import os
if not os.path.exists("data/rag/faiss.index"):
    print("Building FAISS Index...")
    !python main.py --index

from rag.retriever import KidsNutriRetriever
retriever = KidsNutriRetriever()
print("\nTesting Retrieval:")
retriever.debug_retrieve("Can a child eat egg during fever?", top_k=2)


## Section 7: Load Planner
Initialize the Diet Planner and run a verification calculation.

In [ ]:
from planner.diet_planner import KidsNutriDatabase, DietPlanner
db = KidsNutriDatabase()
planner = DietPlanner(db)

profile = {"age": 5, "weight": 18.0, "condition": "healthy_growth", "goal": "balanced_nutrition", "allergies": []}
plan = planner.generate_meal_plan(profile)
print(f"Planner verification successful. Calories calculated: {plan['totals']['calories_kcal']} kcal")


## Section 8: Load LLM
Verify LLM integration scripts.

In [ ]:
print("=== Verifying Gemini API ===")
!python verify_gemini.py

print("\n=== Verifying Groq API ===")
!python verify_groq.py


## Section 9: Single Query Demo
Run the full `main.py --ask` pipeline to display retrieval chunks, planner output, and final latency.

In [ ]:
!python main.py --ask "Can a child with an egg allergy eat boiled eggs?" --age 5 --condition "child_above_1_year" --allergies "egg_protein" --model gemini


## Section 10: Evaluation
Run the benchmark using the decoupled Layer 1 & Layer 2 metrics architecture. (Defaults to 15 questions to save time, change `--num-samples` to 25, 50, or 100 for a full scientific run).

In [ ]:
!python main.py --evaluate --num-samples 15 --models gemini


## Section 11: Metric Reports
Load the generated deterministic metrics.

In [ ]:
import pandas as pd
df = pd.read_csv("reports/final_model_comparison.csv")
display(df)


## Section 12: CSV Reports
Verify all required CSVs were successfully saved to the reports directory.

In [ ]:
import os
reports = os.listdir("reports")
print("Generated Reports:")
for r in sorted(reports):
    if r.endswith(".csv"):
        print(f"- {r}")


## Section 13: Markdown Reports
Display the hallucination and safety analysis reports.

In [ ]:
from IPython.display import Markdown

with open("reports/safety_analysis.md", "r") as f:
    safety = f.read()
    
with open("reports/hallucination_analysis.md", "r") as f:
    hallucination = f.read()
    
display(Markdown("### Safety Extract\n" + safety[:1000] + "...\n\n### Hallucination Extract\n" + hallucination[:1000] + "..."))


## Section 14: Visualizations
Generate all required analytical bar charts.

In [ ]:
import matplotlib.pyplot as plt

df["Hallucination Rate (float)"] = df["Hallucination Rate"].str.rstrip('%').astype(float) / 100.0

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

df.plot(x="Model", y=["MAP@5", "MRR@5", "Context Recall", "Faithfulness", "Answer Relevancy"], kind="bar", ax=ax[0])
ax[0].set_title("RAGAS & Information Retrieval Scores")
ax[0].set_ylabel("Score")
ax[0].legend(loc="lower right")
ax[0].grid(axis="y", linestyle="--", alpha=0.7)

df.plot(x="Model", y=["Safety F2", "Safety F1", "Hallucination Rate (float)"], kind="bar", ax=ax[1], color=["green", "blue", "red"])
ax[1].set_title("Safety F-Scores vs Hallucination")
ax[1].set_ylabel("Percentage / Ratio")
ax[1].grid(axis="y", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.savefig("reports/evaluation_visualizations.png")
plt.show()


## Section 15: Model Comparison
Summarize which model performed best based on empirical metrics.

In [ ]:
best_safety = df.loc[df["Safety F2"].idxmax()]["Model"]
best_faith = df.loc[df["Faithfulness"].idxmax()]["Model"]
lowest_hal = df.loc[df["Hallucination Rate (float)"].idxmin()]["Model"]
fastest = df.loc[df["Average Latency"].idxmin()]["Model"]

print("=== 🏆 FINAL BENCHMARK WINNERS ===")
print(f"Highest Safety (F2): {best_safety}")
print(f"Highest Faithfulness: {best_faith}")
print(f"Lowest Hallucination: {lowest_hal}")
print(f"Fastest Latency: {fastest}")


## Section 16 & 17: Export & Summary
Zip the `reports/` folder for download and officially conclude the experiment.

In [ ]:
import shutil
from IPython.display import FileLink

shutil.make_archive("KidsNutriBite_Reports", 'zip', "reports")
print("Evaluation complete. Reports zipped successfully.")
display(FileLink("KidsNutriBite_Reports.zip"))
